In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
import itertools
import math
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp

DATA = Path.cwd().parents[1] / "data"
OUT = Path.cwd() / "results"

START, END = "2026-02-16", "2026-03-28"
ITERATIONS_PER_DAY = 300
SEED = 0
OUT_PREFIX = "alns3"

DECISIONS = OUT / f"{OUT_PREFIX}_decisions.csv"
ROUTES = OUT / f"{OUT_PREFIX}_routes.csv"
COMPARE_HISTORICAL = True
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

SCENARIO = "S1"
ITERATIONS = 5000
SEEDS = 5
LOST_TRIP_COST = 200.0
BUDGET = {"Fresh": 270, "Style": 480, "Tech": 480}

In [ ]:
DEFAULTS = dict(
    iterations=2000, seed=0,
    fixed_cost=20.0, fuel_cost=1.0, late_penalty=50.0,
    unassigned_penalty=1000.0, history_weight=0.5,
    fresh_start=210, other_start=480, reload_min=0,
    fresh_budget=270, other_budget=480,
    min_remove=2, max_remove=30, max_remove_frac=0.25,
    cooling_rate=0.995, start_temp_frac=0.05, reaction_factor=0.1, segment_length=50,
    sigma1=33, sigma2=13, sigma3=9,
    regret_k=3, tw_regret_weight=100.0, shaw_p=6, worst_p=3,
    shaw_district=9.0, shaw_brand=3.0, shaw_time=3.0, shaw_volume=2.0,
)

In [ ]:
@dataclass(eq=False)
class Order:
    id: str
    outlet: str
    brand: str
    chilled: bool
    district: str
    depot: str
    w: float
    v: float
    s: float
    e: float
    l: float
    van_only: bool
    hist: float = 0.0

In [ ]:
@dataclass(eq=False)
class Vehicle:
    id: str
    type: str
    temp: str
    depot: str
    W: float
    V: float
    km_per_l: float
    km_left: float

In [ ]:
class Solution:
    __slots__ = ("routes", "vcost", "late", "U")

    def __init__(self, n_veh):
        self.routes = [[] for _ in range(n_veh)]
        self.vcost = [0.0] * n_veh
        self.late = [0.0] * n_veh
        self.U = set()

    def copy(self):
        s = Solution(0)
        s.routes = [[t[:] for t in r] for r in self.routes]
        s.vcost, s.late, s.U = self.vcost[:], self.late[:], set(self.U)
        return s

In [ ]:
class ALNS:
    def __init__(self, orders, vehicles, travel, prm=None):
        self.O, self.V, self.tr = orders, vehicles, travel
        self.p = dict(DEFAULTS, **(prm or {}))
        self.rng = random.Random(self.p["seed"])
        self.compat = [[k for k, veh in enumerate(vehicles)
                        if veh.depot == o.depot and (not o.chilled or veh.temp == "reefer")
                        and (not o.van_only or veh.type == "van")
                        and o.w <= veh.W and o.v <= veh.V] for o in orders]
        self.pen = [self.p["unassigned_penalty"] * (1 + self.p["history_weight"] * o.hist) for o in orders]
        self.max_v = max((o.v for o in orders), default=1.0) or 1.0
        self.destroy_ops = {"random": self.destroy_random, "shaw": self.destroy_shaw, "worst": self.destroy_worst}
        self.repair_ops = {"greedy": self.repair_greedy, "regret": self.repair_regret,
                           "tw_regret": self.repair_tw_regret}

    def schedule(self, trips):
        return sorted(trips, key=lambda t: self.O[t[0]].brand != "Fresh")

    def eval_vehicle(self, k, trips):
        if not trips:
            return 0.0, 0.0, {}
        if len(trips) > 2:
            return None
        p, O, veh = self.p, self.O, self.V[k]
        km = late = fresh_min = other_min = 0.0
        clock, arr = None, {}
        for trip in self.schedule(trips):
            first = O[trip[0]]
            brand, dist = first.brand, first.district
            w = v = handling = 0.0
            for i in trip:
                o = O[i]
                if o.brand != brand or o.district != dist:
                    return None
                w += o.w
                v += o.v
                handling += o.s
            if w > veh.W + 1e-9 or v > veh.V + 1e-9:
                return None
            d = self.tr[dist]
            minutes = d["depot_to_district_freeflow_min"] + d["inter_stop_freeflow_min"] * (len(trip) - 1) + handling
            if brand == "Fresh":
                fresh_min += minutes
            else:
                other_min += minutes
            start = p["fresh_start"] if brand == "Fresh" else p["other_start"]
            t = (start if clock is None else max(start, clock + p["reload_min"])) + d["depot_to_district_freeflow_min"]
            prev = None
            for i in trip:
                o = O[i]
                if prev is not None and o.outlet != prev:
                    t += d["inter_stop_freeflow_min"]
                arr[i] = t
                if t > o.l:
                    late += t - o.l
                t = max(t, o.e) + o.s
                prev = o.outlet
            clock = t + d["depot_to_district_freeflow_min"]
            stops = len({O[i].outlet for i in trip})
            km += 2 * d["depot_to_district_km"] + d["inter_stop_km"] * (stops - 1)
        if fresh_min > p["fresh_budget"] + 1e-9 or other_min > p["other_budget"] + 1e-9:
            return None
        if km > veh.km_left + 1e-9:
            return None
        cost = p["fixed_cost"] + p["fuel_cost"] * km / veh.km_per_l + p["late_penalty"] * late
        return cost, late, arr

    def cost(self, sol):
        return sum(sol.vcost) + sum(self.pen[i] for i in sol.U)

    def refresh(self, sol, k):
        r = self.eval_vehicle(k, sol.routes[k])
        assert r is not None, "vehicle became infeasible"
        sol.vcost[k], sol.late[k] = r[0], r[1]

    def locate(self, sol):
        return {i: (k, ti) for k, r in enumerate(sol.routes) for ti, t in enumerate(r) for i in t}

    def remove(self, sol, i, where):
        k, ti = where[i]
        trip = sol.routes[k][ti]
        trip.remove(i)
        if not trip:
            del sol.routes[k][ti]
            for j, (kk, tj) in list(where.items()):
                if kk == k and tj > ti:
                    where[j] = (kk, tj - 1)
        del where[i]
        self.refresh(sol, k)

    def best_in_vehicle(self, sol, i, k):
        o, trips, base = self.O[i], sol.routes[k], sol.vcost[k]
        best = None
        for ti, trip in enumerate(trips):
            f = self.O[trip[0]]
            if f.brand != o.brand or f.district != o.district:
                continue
            for pos in range(len(trip) + 1):
                new = [t if j != ti else t[:pos] + [i] + t[pos:] for j, t in enumerate(trips)]
                r = self.eval_vehicle(k, new)
                if r and (best is None or r[0] - base < best[0]):
                    best = (r[0] - base, ti, pos, o.l - r[2][i])
        if len(trips) < 2:
            r = self.eval_vehicle(k, trips + [[i]])
            if r and (best is None or r[0] - base < best[0]):
                best = (r[0] - base, -1, 0, o.l - r[2][i])
        return best

    def insert(self, sol, i, k, ti, pos):
        if ti == -1:
            sol.routes[k].append([i])
        else:
            sol.routes[k][ti].insert(pos, i)
        self.refresh(sol, k)

    def _repair(self, sol, pool, choose):
        pool = list(pool)
        opts = {i: {k: self.best_in_vehicle(sol, i, k) for k in self.compat[i]} for i in pool}
        while pool:
            i = choose(pool, opts)
            if i is None:
                break
            k = min((k for k, b in opts[i].items() if b), key=lambda k: opts[i][k][0])
            _, ti, pos, _ = opts[i][k]
            self.insert(sol, i, k, ti, pos)
            pool.remove(i)
            for j in pool:
                if k in opts[j]:
                    opts[j][k] = self.best_in_vehicle(sol, j, k)
        sol.U |= set(pool)
        return sol

    def _sorted_costs(self, i, opts):
        return sorted(b[0] for b in opts[i].values() if b)

    def repair_greedy(self, sol, pool):
        def choose(pool, opts):
            best, best_c = None, None
            for i in pool:
                c = self._sorted_costs(i, opts)
                if c and c[0] < self.pen[i] and (best is None or c[0] < best_c):
                    best, best_c = i, c[0]
            return best
        return self._repair(sol, pool, choose)

    def repair_regret(self, sol, pool):
        kk = self.p["regret_k"]

        def choose(pool, opts):
            best, best_key = None, None
            for i in pool:
                c = self._sorted_costs(i, opts)
                if not c or c[0] >= self.pen[i]:
                    continue
                c += [self.pen[i]] * (kk - len(c))
                key = (sum(c[h] - c[0] for h in range(1, kk)), -c[0])
                if best is None or key > best_key:
                    best, best_key = i, key
            return best
        return self._repair(sol, pool, choose)

    def repair_tw_regret(self, sol, pool):
        gamma = self.p["tw_regret_weight"]

        def choose(pool, opts):
            best, best_key = None, None
            for i in pool:
                feas = sorted((b for b in opts[i].values() if b), key=lambda b: b[0])
                if not feas or feas[0][0] >= self.pen[i]:
                    continue
                c1 = feas[0][0]
                c2 = feas[1][0] if len(feas) > 1 else self.pen[i]
                o = self.O[i]
                width = max(o.l - o.e, 1.0)
                urgency = 1 - min(max(feas[0][3] / width, 0.0), 1.0)
                key = (c2 - c1 + gamma * urgency, -c1)
                if best is None or key > best_key:
                    best, best_key = i, key
            return best
        return self._repair(sol, pool, choose)

    def destroy_random(self, sol, n):
        where = self.locate(sol)
        chosen = self.rng.sample(list(where), min(n, len(where)))
        for i in chosen:
            self.remove(sol, i, where)
        return chosen

    def relatedness(self, i, j, where):
        a, b, p = self.O[i], self.O[j], self.p
        return (p["shaw_district"] * (a.district != b.district) + p["shaw_brand"] * (a.brand != b.brand)
                + p["shaw_time"] * abs(a.e - b.e) / 1440 + p["shaw_volume"] * abs(a.v - b.v) / self.max_v
                + (where.get(i) != where.get(j)))

    def destroy_shaw(self, sol, n):
        where = self.locate(sol)
        if not where:
            return []
        static = dict(where)
        removed = [self.rng.choice(list(where))]
        left = [i for i in where if i != removed[0]]
        while len(removed) < n and left:
            r = self.rng.choice(removed)
            left.sort(key=lambda j: self.relatedness(r, j, static))
            idx = int(self.rng.random() ** self.p["shaw_p"] * len(left))
            removed.append(left.pop(idx))
        for i in removed:
            self.remove(sol, i, where)
        return removed

    def destroy_worst(self, sol, n):
        where = self.locate(sol)
        saving = {}
        for i, (k, ti) in where.items():
            trips = [t if j != ti else [x for x in t if x != i] for j, t in enumerate(sol.routes[k])]
            r = self.eval_vehicle(k, [t for t in trips if t])
            saving[i] = sol.vcost[k] - r[0]
        order = sorted(saving, key=lambda i: -saving[i])
        chosen = []
        while order and len(chosen) < n:
            idx = int(self.rng.random() ** self.p["worst_p"] * len(order))
            chosen.append(order.pop(idx))
        for i in chosen:
            self.remove(sol, i, where)
        return chosen

    def initial(self):
        sol = Solution(len(self.V))
        return self.repair_regret(sol, range(len(self.O)))

    def roulette(self, weights):
        total = sum(weights.values())
        x = self.rng.random() * total
        for op, w in weights.items():
            x -= w
            if x <= 0:
                return op
        return op

    def run(self, iterations=None, log=None):
        p = self.p
        iterations = p["iterations"] if iterations is None else iterations
        cur = self.initial()
        best, cur_c = cur, self.cost(cur)
        best_c = cur_c
        T = max(p["start_temp_frac"] * max(sum(cur.vcost), 1.0) / math.log(2), 1e-6)
        wd = {op: 1.0 for op in self.destroy_ops}
        wr = {op: 1.0 for op in self.repair_ops}
        sd, sr = {op: 0.0 for op in wd}, {op: 0.0 for op in wr}
        cd, cr = {op: 0 for op in wd}, {op: 0 for op in wr}
        history = [(0, cur_c, best_c, "", "", "initial")]
        n_assigned = len(self.O) - len(cur.U)
        for it in range(1, iterations + 1):
            d_op, r_op = self.roulette(wd), self.roulette(wr)
            hi = max(p["min_remove"], min(p["max_remove"], int(p["max_remove_frac"] * max(n_assigned, 1))))
            n_remove = self.rng.randint(p["min_remove"], hi)
            cand = cur.copy()
            removed = self.destroy_ops[d_op](cand, n_remove)
            pool = removed + list(cand.U)
            cand.U = set()
            self.repair_ops[r_op](cand, pool)
            cand_c = self.cost(cand)

            delta = cand_c - cur_c
            if delta < -1e-9 or self.rng.random() < math.exp(-delta / T):
                cur, cur_c = cand, cand_c
                n_assigned = len(self.O) - len(cur.U)
                if cand_c < best_c - 1e-9:
                    best, best_c, score, outcome = cand, cand_c, p["sigma1"], "best"
                elif delta < -1e-9:
                    score, outcome = p["sigma2"], "better"
                else:
                    score, outcome = p["sigma3"], "accepted"
            else:
                score, outcome = 0, "rejected"
            sd[d_op] += score
            sr[r_op] += score
            cd[d_op] += 1
            cr[r_op] += 1
            if it % p["segment_length"] == 0:
                for w, s, c in ((wd, sd, cd), (wr, sr, cr)):
                    for op in w:
                        if c[op]:
                            w[op] = (1 - p["reaction_factor"]) * w[op] + p["reaction_factor"] * s[op] / c[op]
                            s[op], c[op] = 0.0, 0
            T *= p["cooling_rate"]
            history.append((it, cur_c, best_c, d_op, r_op, outcome))
            if log and it % log == 0:
                print(f"  iter {it}: current {cur_c:,.1f}, best {best_c:,.1f}, unassigned {len(best.U)}")
        self.weights = {"destroy": wd, "repair": wr}
        return best, history

    def assignments(self, sol):
        out = {}
        for k, trips in enumerate(sol.routes):
            if not trips:
                continue
            arr = self.eval_vehicle(k, trips)[2]
            for n, trip in enumerate(self.schedule(trips), start=1):
                for i in trip:
                    out[i] = dict(vehicle_id=self.V[k].id, trip_id=n, arrival_min=arr[i],
                                  late_min=max(0.0, arr[i] - self.O[i].l))
        return out

    def trip_km(self, trip):
        d = self.tr[self.O[trip[0]].district]
        return 2 * d["depot_to_district_km"] + d["inter_stop_km"] * (len({self.O[i].outlet for i in trip}) - 1)

In [ ]:
def hhmm_to_min(s):
    h, m = map(int, str(s).split(":"))
    return h * 60 + m

In [ ]:
def min_to_hhmm(x):
    return f"{int(x) // 60:02d}:{int(x) % 60:02d}"

In [ ]:
DEFAULTS2 = DEFAULTS
DEFAULTS = dict(DEFAULTS2, lost_trip_cost=200.0, rightsize_passes=3)

In [ ]:
class ALNS3(ALNS):
    def __init__(self, orders, vehicles, travel, prm=None):
        super().__init__(orders, vehicles, travel, dict(DEFAULTS, **(prm or {})))
        self.compat_set = [set(c) for c in self.compat]
        self.model = [(v.depot, v.type, v.temp, v.W, v.V, v.km_per_l) for v in vehicles]
        self.depot_vmax = {}
        for v in vehicles:
            self.depot_vmax[v.depot] = max(self.depot_vmax.get(v.depot, 0.0), v.V)
        min_service = min((o.s for o in orders if o.brand == "Fresh"), default=15.0)
        self.min_extra_fresh = {}
        for d, row in travel.items():
            dep = row["depot"]
            t = row["depot_to_district_freeflow_min"] + min_service
            self.min_extra_fresh[dep] = min(self.min_extra_fresh.get(dep, t), t)
        chilled_m3, fresh_m3, reefer_m3, fleet_m3 = {}, {}, {}, {}
        for o in orders:
            if o.brand == "Fresh":
                fresh_m3[o.depot] = fresh_m3.get(o.depot, 0.0) + o.v
                if o.chilled:
                    chilled_m3[o.depot] = chilled_m3.get(o.depot, 0.0) + o.v
        for v in vehicles:
            fleet_m3[v.depot] = fleet_m3.get(v.depot, 0.0) + v.V
            if v.temp == "reefer":
                reefer_m3[v.depot] = reefer_m3.get(v.depot, 0.0) + v.V
        self.scarcity = []
        for v in vehicles:
            demand, cap = ((chilled_m3, reefer_m3) if v.temp == "reefer" else (fresh_m3, fleet_m3))
            ratio = demand.get(v.depot, 0.0) / max(cap.get(v.depot, 0.0), 1e-9)
            self.scarcity.append(min(max(ratio - 1.0, 0.0), 1.0))
        self.destroy_ops["district"] = self.destroy_district
        for name, op in list(self.repair_ops.items()):
            self.repair_ops[name] = (lambda op: lambda sol, pool: self.rightsize(op(sol, pool)))(op)

    def trip_minutes(self, trip):
        d = self.tr[self.O[trip[0]].district]
        return (d["depot_to_district_freeflow_min"] + d["inter_stop_freeflow_min"] * (len(trip) - 1)
                + sum(self.O[i].s for i in trip))

    def eval_vehicle(self, k, trips):
        r = super().eval_vehicle(k, trips)
        if r is None or not self.scarcity[k]:
            return r
        fresh = [t for t in trips if self.O[t[0]].brand == "Fresh"]
        if len(fresh) != 1:
            return r
        veh = self.V[k]
        if self.p["fresh_budget"] - self.trip_minutes(fresh[0]) < self.min_extra_fresh.get(veh.depot, 0):
            lost = self.p["lost_trip_cost"] * self.scarcity[k] * veh.V / self.depot_vmax[veh.depot]
            return r[0] + lost, r[1], r[2]
        return r

    def candidates(self, sol, i):
        out, reps = [], {}
        for k in self.compat[i]:
            if sol.routes[k]:
                out.append(k)
            else:
                m = self.model[k]
                if m not in reps or self.V[k].km_left > self.V[reps[m]].km_left:
                    reps[m] = k
        return out + list(reps.values())

    def _repair(self, sol, pool, choose):
        pool = list(pool)
        opts = {i: {k: self.best_in_vehicle(sol, i, k) for k in self.candidates(sol, i)} for i in pool}
        while pool:
            i = choose(pool, opts)
            if i is None:
                break
            k = min((k for k, b in opts[i].items() if b), key=lambda k: opts[i][k][0])
            _, ti, pos, _ = opts[i][k]
            self.insert(sol, i, k, ti, pos)
            pool.remove(i)
            for j in pool:
                old = opts[j]
                opts[j] = {c: (old[c] if c in old and c != k else self.best_in_vehicle(sol, j, c))
                           for c in self.candidates(sol, j)}
        sol.U |= set(pool)
        return sol

    def rightsize(self, sol):
        for _ in range(self.p["rightsize_passes"]):
            moved = False
            for k in range(len(self.V)):
                for ti in range(len(sol.routes[k])):
                    trip = sol.routes[k][ti]
                    rest = [t for j, t in enumerate(sol.routes[k]) if j != ti]
                    src = self.eval_vehicle(k, rest)
                    dests = set.intersection(*(self.compat_set[i] for i in trip)) - {k}
                    best, seen = None, set()
                    for d in sorted(dests, key=lambda d: -self.V[d].km_left):
                        if len(sol.routes[d]) >= 2:
                            continue
                        if not sol.routes[d]:
                            if self.model[d] in seen:
                                continue
                            seen.add(self.model[d])
                        r = self.eval_vehicle(d, sol.routes[d] + [trip])
                        if r is None:
                            continue
                        delta = src[0] + r[0] - sol.vcost[k] - sol.vcost[d]
                        if delta < -1e-6 and (best is None or delta < best[0]):
                            best = (delta, d)
                    if best:
                        d = best[1]
                        sol.routes[k] = rest
                        sol.routes[d] = sol.routes[d] + [trip]
                        self.refresh(sol, k)
                        self.refresh(sol, d)
                        moved = True
                        break
            if not moved:
                break
        return sol

    def destroy_district(self, sol, n):
        where = self.locate(sol)
        if not where:
            return []
        o = self.O[self.rng.choice(list(where))]
        chosen = [i for i in where if self.O[i].depot == o.depot and self.O[i].district == o.district
                  and self.O[i].brand == o.brand]
        for i in chosen:
            self.remove(sol, i, where)
        return chosen

    def plan_cost(self, sol):
        total = sum(self.pen[i] for i in sol.U)
        for k, trips in enumerate(sol.routes):
            if trips:
                total += ALNS.eval_vehicle(self, k, trips)[0]
        return total

In [ ]:
raw = pd.concat([pd.read_csv(DATA / "Training Data/deliveries_train.csv"),
                 pd.read_csv(DATA / "Test Data/task1_test_inputs.csv")], ignore_index=True)
raw = raw[(raw.order_date >= START) & (raw.order_date <= END)]
outlets = pd.read_csv(DATA / "General Data/outlets.csv").set_index("outlet_id")
vehicles = pd.read_csv(DATA / "General Data/vehicles.csv").rename(columns=str.strip)
if "weekly_range_km" not in vehicles:
    vehicles["weekly_range_km"] = vehicles.km_per_l * vehicles.weekly_fuel_quota_l
travel = pd.read_csv(DATA / "General Data/district_travel.csv").set_index("district").to_dict("index")
service = pd.read_csv(DATA / "General Data/service_allowance.csv").set_index(["brand", "dock_type"]).service_allowance_min

by_date = defaultdict(list)
for r in raw.itertuples():
    out = outlets.loc[r.outlet_id]
    by_date[r.order_date].append(dict(
        order=Order(r.delivery_id, r.outlet_id, r.brand, r.temp_requirement == "chilled", r.district, r.depot,
                    r.order_weight_kg, r.order_volume_m3, service[(r.brand, out.dock_type)],
                    hhmm_to_min(out.window_open_time), hhmm_to_min(out.window_close_time),
                    out.parking_constraint == "van_only"),
        order_date=r.order_date, times_deferred=0))
dates = sorted(by_date)
category = lambda o: ("fresh_chilled" if o.chilled else "fresh_dry") if o.brand == "Fresh" else o.brand.lower()
print(f"{len(dates)} days, {len(raw):,} orders")

In [ ]:
week_used = {v: 0.0 for v in vehicles.vehicle_id}
outlet_deferrals = defaultdict(deque)
pending, week, decisions, routes = [], None, [], []
t_start = time.time()
for n, date in enumerate(dates, 1):
    day = pd.Timestamp(date)
    if day.isocalendar()[:2] != week:
        week = day.isocalendar()[:2]
        week_used = {v: 0.0 for v in week_used}
    todays = pending + by_date[date]
    for item in todays:
        hist = outlet_deferrals[item["order"].outlet]
        while hist and hist[0] < day - pd.Timedelta(days=30):
            hist.popleft()
        item["order"].hist = len(hist)
    vehs = [Vehicle(r.vehicle_id, r.type, r.temp, r.depot, r.weight_cap_kg, r.volume_cap_m3, r.km_per_l,
                    r.weekly_range_km - week_used[r.vehicle_id]) for r in vehicles.itertuples()]
    alns = ALNS3([it["order"] for it in todays], vehs, travel, dict(iterations=ITERATIONS_PER_DAY, seed=SEED + n))
    best, _ = alns.run()
    asg = alns.assignments(best)

    for k, trips in enumerate(best.routes):
        for t_no, trip in enumerate(alns.schedule(trips), start=1):
            os_, veh = [alns.O[i] for i in trip], vehs[k]
            km = alns.trip_km(trip)
            week_used[veh.id] += km
            cats = {category(o) for o in os_}
            w, v = sum(o.w for o in os_), sum(o.v for o in os_)
            routes.append(dict(plan_date=date, route_id=f"{date}-{veh.id}-T{t_no}", depot=veh.depot, vehicle_id=veh.id,
                               vehicle_type=veh.type, vehicle_temp=veh.temp, trip_id=t_no, district=os_[0].district,
                               category=cats.pop() if len(cats) == 1 else "fresh_mixed", orders=len(trip),
                               stops=len({o.outlet for o in os_}), weight_kg=round(w, 1), volume_m3=round(v, 3),
                               weight_util=round(w / veh.W, 4), volume_util=round(v / veh.V, 4), km=round(km, 1),
                               late_min=round(sum(asg[i]["late_min"] for i in trip), 1)))
    deferred = []
    for i, item in enumerate(todays):
        o, served = item["order"], i in asg
        decisions.append(dict(plan_date=date, delivery_id=o.id, order_date=item["order_date"], outlet_id=o.outlet,
                              brand=o.brand, category=category(o), district=o.district, depot=o.depot,
                              van_only=o.van_only, weight_kg=o.w, volume_m3=o.v,
                              times_deferred_before=item["times_deferred"], p=o.hist,
                              decision="served" if served else "deferred",
                              vehicle_id=asg[i]["vehicle_id"] if served else "",
                              trip_id=asg[i]["trip_id"] if served else "",
                              route_id=f"{date}-{asg[i]['vehicle_id']}-T{asg[i]['trip_id']}" if served else "",
                              late_min=round(asg[i]["late_min"], 1) if served else ""))
        if not served:
            item["times_deferred"] += 1
            outlet_deferrals[o.outlet].append(day)
            deferred.append(item)
    pending = deferred
    if n % 5 == 0 or n == len(dates):
        print(f"{date}: day {n}/{len(dates)}, {len(todays)} orders, {len(deferred)} deferred ({time.time() - t_start:.0f} s)")

dec, rts = pd.DataFrame(decisions), pd.DataFrame(routes)
print(f"{(dec.decision == 'deferred').sum():,} deferrals, {len(pending)} still pending, "
      f"{(pd.to_numeric(dec.late_min, errors='coerce') > 0).sum():,} late deliveries; {len(rts):,} vehicle-trips")

In [ ]:
OUT.mkdir(exist_ok=True)
dec.to_csv(OUT / f"{OUT_PREFIX}_decisions.csv", index=False)
rts.to_csv(OUT / f"{OUT_PREFIX}_routes.csv", index=False)
kpl = vehicles.set_index("vehicle_id").km_per_l
per_day = rts.assign(litres=rts.km / rts.vehicle_id.map(kpl)).groupby("plan_date")
pd.Series({"trips per day": len(rts) / rts.plan_date.nunique(), "vehicles per day": per_day.vehicle_id.nunique().mean(),
           "km per day": per_day.km.sum().mean(), "litres per day": per_day.litres.sum().mean()}).round(1)

In [ ]:
dec = pd.read_csv(DECISIONS)
all_outlets = pd.read_csv(DATA / "General Data/outlets.csv").outlet_id
runs = {"algorithm": dec}

In [ ]:
if COMPARE_HISTORICAL:
    h = pd.concat([pd.read_csv(DATA / "Training Data/deliveries_train.csv"),
                   pd.read_csv(DATA / "Test Data/task1_test_inputs.csv")], ignore_index=True)
    h["category"] = h.brand.str.lower().where(h.brand != "Fresh",
                                              "fresh_" + h.temp_requirement.map({"chilled": "chilled", "ambient": "dry"}))
    hist_days = sorted(h.order_date.unique())
    idx = {d: i for i, d in enumerate(hist_days)}
    h = h[h.delivery_id.isin(set(dec.delivery_id))]
    rows = []
    for r in h.itertuples():
        start = idx[r.order_date]
        if r.dispatch_status == "attempted":
            end = idx.get(r.dispatch_date, start)
        elif r.dispatch_status == "deferred":
            end = idx.get(r.dispatch_date, len(hist_days))
        else:
            end = start + 1
        base = dict(delivery_id=r.delivery_id, outlet_id=r.outlet_id, category=r.category, district=r.district)
        for i in range(start, min(end, len(hist_days))):
            rows.append(dict(base, plan_date=hist_days[i], decision="deferred"))
        if r.dispatch_status != "not_run" and end < len(hist_days):
            rows.append(dict(base, plan_date=hist_days[end], decision="served"))
    runs["historical"] = pd.DataFrame(rows)

In [ ]:
summary, tables = {}, {}
for name, d in runs.items():
    orders = d.groupby("delivery_id").agg(outlet_id=("outlet_id", "first"), category=("category", "first"),
                                          district=("district", "first"),
                                          deferrals=("decision", lambda s: (s == "deferred").sum()),
                                          served=("decision", lambda s: (s == "served").any()))
    day_index = {x: i for i, x in enumerate(sorted(d.plan_date.unique()))}
    ev = d[d.decision == "deferred"][["delivery_id", "plan_date"]].copy()
    if ev.empty:
        orders["longest_run"] = 0
    else:
        ev["i"] = ev.plan_date.map(day_index)
        ev = ev.sort_values(["delivery_id", "i"])
        ev["run_id"] = ((ev.i.diff() != 1) | (ev.delivery_id != ev.delivery_id.shift())).cumsum()
        orders["longest_run"] = ev.groupby(["delivery_id", "run_id"]).size().groupby("delivery_id").max()
        orders["longest_run"] = orders.longest_run.fillna(0).astype(int)
    events = d[d.decision == "deferred"]
    per_outlet = events.groupby("outlet_id").size().reindex(all_outlets, fill_value=0)
    x = np.sort(per_outlet.values.astype(float))
    gini = float((2 * np.arange(1, len(x) + 1) - len(x) - 1).dot(x) / (len(x) * x.sum())) if x.sum() else 0.0
    deferred_orders = orders[orders.deferrals > 0]
    summary[name] = {
        "orders": len(orders),
        "deferral events (order-days)": len(events),
        "orders ever deferred": len(deferred_orders),
        "share of orders deferred": f"{len(deferred_orders) / len(orders):.1%}",
        "orders never served": int((~orders.served).sum()),
        "longest continuous deferral (days)": int(orders.longest_run.max()),
        "mean run for deferred orders (days)": round(deferred_orders.longest_run.mean(), 2) if len(deferred_orders) else 0,
        "orders deferred 2+ days in a row": int((orders.longest_run >= 2).sum()),
        "orders deferred 3+ days in a row": int((orders.longest_run >= 3).sum()),
        "outlets with any deferral": f"{int((per_outlet > 0).sum())} of {len(all_outlets)}",
        "deferrals at the worst outlet": int(per_outlet.max()),
        "share of deferrals at the worst 5 outlets": f"{per_outlet.nlargest(5).sum() / per_outlet.sum():.1%}"
        if per_outlet.sum() else "0.0%",
        "Gini of deferrals across outlets": round(gini, 2),
    }
    run_dist = deferred_orders.longest_run.clip(upper=5).value_counts().sort_index()
    run_dist.index = [f"{i}+" if i == 5 else str(i) for i in run_dist.index]
    tables[name] = {
        "by category": events.groupby("category").size(),
        "by district": events.groupby("district").size(),
        "by weekday": events.groupby(pd.to_datetime(events.plan_date).dt.dayofweek).size().rename(index=dict(enumerate(DAYS))),
        "by longest continuous run (days)": run_dist,
        "worst 10 outlets": per_outlet[per_outlet > 0].nlargest(10),
    }
pd.DataFrame(summary)

In [ ]:
for title in tables["algorithm"]:
    t = pd.concat({n: tables[n][title] for n in runs}, axis=1).fillna(0).astype(int)
    print(f"\nDeferral events {title}\n{t.to_string() if len(t) else 'none'}")

In [ ]:
if ROUTES.exists():
    rts = pd.read_csv(ROUTES)
    per_day = rts.groupby("plan_date")
    print(pd.Series({
        "vehicle-trips per day": len(rts) / rts.plan_date.nunique(),
        "vehicles used per day": per_day.vehicle_id.nunique().mean(),
        "vehicles running 2 trips per day": (rts.groupby(["plan_date", "vehicle_id"]).size() == 2).groupby("plan_date").sum().mean(),
        "stops per trip": rts.stops.mean(),
        "km per day": per_day.km.sum().mean(),
    }).round(2).to_string())
    display_fill = rts.groupby("category")[["weight_util", "volume_util"]].median().round(2)
    print(display_fill.to_string())

In [ ]:
s = pd.read_csv(DATA / "Test Data/task2b_peak_day_scenarios.csv")
s = s[s.scenario == SCENARIO].reset_index(drop=True)
fleet = pd.read_csv(DATA / "Test Data/task2b_peak_day_fleet.csv")
vehicles = pd.read_csv(DATA / "General Data/vehicles.csv").rename(columns=str.strip)
if "weekly_range_km" not in vehicles:
    vehicles["weekly_range_km"] = vehicles.km_per_l * vehicles.weekly_fuel_quota_l
vehicles = vehicles.merge(fleet[fleet.scenario == SCENARIO], on="vehicle_id")
travel = pd.read_csv(DATA / "General Data/district_travel.csv").set_index("district")
service = pd.read_csv(DATA / "General Data/service_allowance.csv").set_index(["brand", "dock_type"])

orders = [Order(r.order_ref, r.outlet_id, r.brand, r.temp_requirement == "chilled", r.district, r.depot,
                r.order_weight_kg, r.order_volume_m3, service.loc[(r.brand, r.dock_type), "service_allowance_min"],
                hhmm_to_min(r.window_open_time), hhmm_to_min(r.window_close_time),
                r.parking_constraint == "van_only", hist=r.deferred_yesterday * r.days_since_last_served)
          for r in s.itertuples()]
avail = vehicles[vehicles.status == "available"]
vehs = [Vehicle(r.vehicle_id, r.type, r.temp, r.depot, r.weight_cap_kg, r.volume_cap_m3, r.km_per_l, r.weekly_range_km)
        for r in avail.itertuples()]
tr = travel.to_dict("index")
print(f"{len(orders)} orders, {len(vehs)} available vehicles")

In [ ]:
reefers = avail[avail.temp == "reefer"].reset_index(drop=True)
ch = s[s.temp_requirement == "chilled"].reset_index(drop=True)
cols = []
for vi, v in reefers.iterrows():
    for dist, g in ch[ch.depot == v.depot].groupby("district"):
        ok = [r for r in g.index if not (ch.at[r, "parking_constraint"] == "van_only" and v.type != "van")]
        d = travel.loc[dist]
        for size in range(1, len(ok) + 1):
            for combo in itertools.combinations(ok, size):
                rows = ch.loc[list(combo)]
                if rows.order_weight_kg.sum() > v.weight_cap_kg or rows.order_volume_m3.sum() > v.volume_cap_m3:
                    continue
                minutes = (d.depot_to_district_freeflow_min + d.inter_stop_freeflow_min * (size - 1)
                           + sum(service.loc[("Fresh", dk), "service_allowance_min"] for dk in rows.dock_type))
                if minutes <= BUDGET["Fresh"]:
                    cols.append((vi, combo, minutes))
A, ub = [], []
for vi in range(len(reefers)):
    A.append([1.0 if c[0] == vi else 0.0 for c in cols]); ub.append(2)
    A.append([c[2] if c[0] == vi else 0.0 for c in cols]); ub.append(BUDGET["Fresh"])
for r in ch.index:
    A.append([1.0 if r in c[1] else 0.0 for c in cols]); ub.append(1)
res = milp(-np.array([len(c[1]) for c in cols], dtype=float), integrality=np.ones(len(cols)), bounds=Bounds(0, 1),
           constraints=[LinearConstraint(np.array(A), -np.inf, np.array(ub, dtype=float))])
bound = int(round(-res.fun))
print(f"Upper bound: the available reefers can carry at most {bound} of {len(ch)} chilled orders")
pd.DataFrame([(reefers.at[c[0], "vehicle_id"], ch.at[c[1][0], "district"], len(c[1]), int(c[2]))
              for j, c in enumerate(cols) if res.x[j] > 0.5], columns=["vehicle", "district", "orders", "minutes"])

In [ ]:
rows, best3 = [], None
for seed in range(SEEDS):
    for name, cls, extra in (("algorithm 2", ALNS, {}), ("algorithm 3", ALNS3, {"lost_trip_cost": LOST_TRIP_COST})):
        alns = cls(orders, vehs, tr, dict(extra, iterations=ITERATIONS, seed=seed))
        t0 = time.time()
        sol, hist = alns.run()
        secs = time.time() - t0
        asg = alns.assignments(sol)
        used = [k for k, r in enumerate(sol.routes) if r]
        rows.append(dict(algorithm=name, seed=seed, seconds=round(secs, 1),
                         F=alns.plan_cost(sol) if cls is ALNS3 else alns.cost(sol),
                         served=len(asg), chilled=sum(1 for i, o in enumerate(orders) if o.chilled and i in asg),
                         late=sum(sol.late), vehicles=len(used),
                         litres=sum(alns.trip_km(t) / vehs[k].km_per_l for k in used for t in sol.routes[k]),
                         two_trip_reefers=sum(1 for k in used if vehs[k].temp == "reefer" and len(sol.routes[k]) == 2)))
        print(f"{name} seed {seed}: served {rows[-1]['served']}, chilled {rows[-1]['chilled']}, {secs:.0f} s")
        if cls is ALNS3 and (best3 is None or alns.cost(sol) < best3[0].cost(best3[1])):
            best3 = (alns, sol, hist)
res_df = pd.DataFrame(rows)
res_df.groupby("algorithm").agg(served_mean=("served", "mean"), served_best=("served", "max"),
                                chilled_mean=("chilled", "mean"), chilled_best=("chilled", "max"),
                                F_mean=("F", "mean"), F_best=("F", "min"), litres=("litres", "mean"),
                                vehicles=("vehicles", "mean"), two_trip_reefers=("two_trip_reefers", "mean"),
                                late=("late", "mean"), seconds=("seconds", "mean")).T.round(1)

In [ ]:
res_df.round(1)

In [ ]:
alns, sol, hist = best3
h = pd.DataFrame(hist, columns=["iter", "current", "best", "destroy", "repair", "outcome"])
fig, ax = plt.subplots(figsize=(10, 4.5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.plot(h.iter, h.current, color="#86b6ef", lw=1, label="current")
ax.plot(h.iter, h.best, color="#1c5cab", lw=2, label="best")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#e1e0d9", lw=0.6)
ax.set_xlabel("Iteration", color="#52514e")
ax.set_ylabel("Objective F(S)", color="#52514e")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.legend(frameon=False, labelcolor="#52514e")
ax.set_title(f"Algorithm 3 convergence on Task 2B scenario {SCENARIO} (best seed)", loc="left", color="#0b0b0b")
plt.show()
print("operator weights:", {k: {op: round(w, 2) for op, w in v.items()} for k, v in alns.weights.items()})

In [ ]:
asg = alns.assignments(sol)
sub = pd.DataFrame([dict(scenario=SCENARIO, order_ref=o.id, outlet_id=o.outlet,
                         decision="served" if i in asg else "deferred",
                         vehicle_id=asg[i]["vehicle_id"] if i in asg else "",
                         trip_id=asg[i]["trip_id"] if i in asg else "") for i, o in enumerate(orders)])
OUT.mkdir(exist_ok=True)
sub.to_csv(OUT / "task2b_alns3_submission.csv", index=False)
print(f"served {len(asg)} of {len(orders)}")

v_idx = vehicles.set_index("vehicle_id")
served = sub[sub.decision == "served"].merge(
    s[["order_ref", "brand", "district", "depot", "dock_type", "parking_constraint", "temp_requirement",
       "order_weight_kg", "order_volume_m3"]], on="order_ref")
violations, trip_rows = [], []
if served.order_ref.duplicated().any():
    violations.append("rule 5: an order is assigned more than once")
for (veh, trip), rows in served.groupby(["vehicle_id", "trip_id"]):
    car, tag = v_idx.loc[veh], f"{veh} trip {trip}"
    if car.status != "available":
        violations.append(f"{tag}: vehicle is {car.status}")
    if rows.brand.nunique() > 1 or rows.district.nunique() > 1:
        violations.append(f"rule 1: {tag} mixes brands or districts")
    if (rows.temp_requirement == "chilled").any() and car.temp != "reefer":
        violations.append(f"rule 2: {tag} carries chilled orders on an ambient vehicle")
    if (rows.parking_constraint == "van_only").any() and car.type != "van":
        violations.append(f"rule 3: {tag} serves a van-only outlet with a {car.type}")
    if (rows.depot != car.depot).any():
        violations.append(f"rule 4: {tag} serves another depot's outlets")
    w, vol = rows.order_weight_kg.sum(), rows.order_volume_m3.sum()
    if w > car.weight_cap_kg + 1e-9 or vol > car.volume_cap_m3 + 1e-9:
        violations.append(f"rule 6: {tag} over capacity ({w:,.0f}/{car.weight_cap_kg:,} kg, {vol:.2f}/{car.volume_cap_m3} m³)")
    d = travel.loc[rows.district.iloc[0]]
    minutes = (d.depot_to_district_freeflow_min + d.inter_stop_freeflow_min * (len(rows) - 1)
               + sum(service.loc[(b, dk), "service_allowance_min"] for b, dk in zip(rows.brand, rows.dock_type)))
    trip_rows.append(dict(vehicle_id=veh, trip_id=trip, vehicle_type=car.type, vehicle_temp=car.temp,
                          brand=rows.brand.iloc[0], district=rows.district.iloc[0], orders=len(rows),
                          chilled_orders=int((rows.temp_requirement == "chilled").sum()),
                          weight_util=round(w / car.weight_cap_kg, 3), volume_util=round(vol / car.volume_cap_m3, 3),
                          minutes=int(minutes)))
trips = pd.DataFrame(trip_rows).sort_values(["vehicle_id", "trip_id"])
for veh, rows in trips.groupby("vehicle_id"):
    if len(rows) > 2 or sorted(rows.trip_id) not in ([1], [1, 2]):
        violations.append(f"rule 7: {veh} has trips {sorted(rows.trip_id)}")
    fresh_min = rows.loc[rows.brand == "Fresh", "minutes"].sum()
    other_min = rows.loc[rows.brand != "Fresh", "minutes"].sum()
    if fresh_min > BUDGET["Fresh"]:
        violations.append(f"rule 7: {veh} Fresh trips take {fresh_min} min (budget 270)")
    if other_min > BUDGET["Style"]:
        violations.append(f"rule 7: {veh} Style/Tech trips take {other_min} min (budget 480)")
print("All booklet rules pass." if not violations else f"{len(violations)} violations:\n" + "\n".join(violations))
trips

In [ ]:
pd.DataFrame([dict(order_ref=o.id, outlet=o.outlet, district=o.district,
                   category="chilled" if o.chilled else o.brand, volume_m3=round(o.v, 2), weight_kg=round(o.w))
              for i, o in enumerate(orders) if i not in asg])